# Building a RAG System with Gemini Embeddings, ChromaDB, and Groq

## What this notebook builds

This notebook follows the teaching sequence of the instructor's **Building a RAG System with LangChain and ChromaDB** notebook, while replacing OpenAI API usage with the provider combination requested for this class:

- **Gemini API (`google-genai`)** creates document and query embeddings.
- **ChromaDB + LangChain Chroma** stores vectors with text and metadata, then retrieves relevant chunks.
- **Groq API (`langchain-groq`)** runs the chat model that generates answers from retrieved context.
- **LangChain** connects documents, splitting, embedding interfaces, retrievers, prompts, and chains.

**Provider separation is intentional:** Gemini cells handle embeddings; Groq cells handle text generation. There is no OpenAI API key or OpenAI API client in this notebook. The selected generation model is hosted by **Groq** and called through `ChatGroq`; only `GROQ_API_KEY` is required for generation. The embedding and chat models are deliberately isolated in separate sections.

## Learning goals

Explain and implement document loading, chunking, embeddings, persistent vector storage, similarity search, distance scores, metadata filters, retrievers, prompt templates, document chains, retrieval chains, LCEL, adding documents, and conversational query reformulation.

For each stage, Markdown explains **what we do, why we do it, which component we use, how it works, and what to inspect**. Run code cells from top to bottom.

## Main architecture

`Text files → Documents → Chunks → Gemini embeddings → Cached embeddings → ChromaDB → Retriever → Prompt + retrieved context → Groq model → Answer + sources`

> **Execution note:** API smoke tests and uncached embedding calls use your API quotas. This notebook has been statically checked, but external API cells have not been executed with your credentials. Never commit `.env`, API keys, or private data.


## Provider responsibilities: keep the roles separate

| Component | Responsibility | Not responsible for |
|---|---|---|
| Gemini Embedding 2 | Maps a document chunk or query into a vector | Writing the final answer |
| ChromaDB | Persists vectors, text, IDs, and metadata; searches for nearby vectors | Generating the final answer |
| Groq-hosted chat model | Generates an answer using the prompt and retrieved context | Searching ChromaDB by itself |
| LangChain | Connects components through standard interfaces | Guaranteeing answer correctness |

**Interview definition — RAG:** Retrieval-Augmented Generation is an architecture that retrieves relevant external information at query time and supplies it as context to a generative model. It can improve grounding, but retrieval errors, incomplete documents, and model mistakes can still produce incorrect answers.


## 1. Prepare the Python environment

### What and why?

Install the separate packages needed for the Gemini API, Groq chat integration, LangChain components, ChromaDB, text splitting, and environment variables. Use the same environment as your Jupyter kernel.

### Windows CMD / project terminal

Run this command in the project terminal, **not in a Python code cell**:

```cmd
uv add google-genai langchain-core langchain-community langchain-text-splitters langchain-chroma chromadb langchain-groq langchain-classic python-dotenv
```

`langchain-classic` is included to preserve the instructor's `create_stuff_documents_chain`, `create_retrieval_chain`, history-aware retriever, and cache-backed embedding lessons. The notebook also contains an LCEL alternative so students can compare both patterns.


## 2. Store API keys safely

Create a `.env` file in the project directory or a parent directory:

```env
GEMINI_API_KEY=your_gemini_api_key_here
GROQ_API_KEY=your_groq_api_key_here
```

Replace placeholders locally. Do not write real keys into notebook source, print them, or commit `.env` to Git. Add `.env`, local ChromaDB directories, and embedding-cache directories to `.gitignore`. The next cell checks only whether keys exist; it never displays their values.


In [1]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Search from the current working directory and its parent folders.
env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
else:
    # Also supports environment variables already set in the shell/kernel.
    load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

missing_keys = [
    name
    for name, value in (
        ("GEMINI_API_KEY", GEMINI_API_KEY),
        ("GROQ_API_KEY", GROQ_API_KEY),
    )
    if not value
]

if missing_keys:
    raise EnvironmentError(
        "Missing environment variable(s): " + ", ".join(missing_keys)
        + ". Add them to .env or set them in the environment."
    )

print("Gemini API key found; value hidden.")
print("Groq API key found; value hidden.")
print(".env path:", env_path if env_path else "not found; shell variables may be in use")

Gemini API key found; value hidden.
Groq API key found; value hidden.
.env path: e:\RAG\.env


## 3. Import the required libraries

- `Path` and `os`: filesystem paths and environment settings.
- `genai`: direct access to Gemini's embedding endpoint.
- `Document` and `Embeddings`: LangChain's standard document and embedding interfaces.
- `DirectoryLoader` / `TextLoader`: load multiple text files.
- `RecursiveCharacterTextSplitter`: split documents into chunks.
- `Chroma` from `langchain-chroma`: persistent vector-store and search integration.
- `ChatGroq`: LangChain interface for Groq-hosted chat models.
- Prompts, messages, output parsers, and chain helpers: build answer and conversational flows.

The dedicated `langchain-chroma` package is used rather than the older community import path. No OpenAI client or OpenAI embeddings class is imported.


In [2]:
from google import genai
import chromadb

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

from langchain_classic.chains import (
    create_history_aware_retriever,
    create_retrieval_chain,
)
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

print("Required imports completed.")

C:\Users\sniti\AppData\Local\Temp\ipykernel_26108\1985218772.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Required imports completed.


## 4. Initialize Gemini for embeddings only

A **client** is the configured interface Python uses to send requests to a provider. We initialize Gemini separately from Groq so authentication or embedding errors can be isolated before we involve the vector store or answer-generation model.

`gemini-embedding-2` is Google's stable Gemini Embedding 2 model ID in the current documentation. Keep the model and query/document formatting consistent for every vector stored in a collection. If the model or formatting changes, rebuild vectors and use a new cache namespace.


In [3]:
GEMINI_EMBEDDING_MODEL = "gemini-embedding-2"

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini embedding client initialized.")
print("Embedding model:", GEMINI_EMBEDDING_MODEL)

Gemini embedding client initialized.
Embedding model: gemini-embedding-2


## 5. Test one Gemini embedding directly

An **embedding** is a numeric vector produced from content. It is useful for measuring semantic relationships; it is not a generated answer.

This direct API example preserves the instructor's standalone embedding demonstration. It verifies the API response before we wrap Gemini in LangChain's `Embeddings` interface.

For text-only question-answering retrieval, Google's guidance recommends the query format `task: question answering | query: ...` and document format `title: none | text: ...`. These formats intentionally differ. Gemini Embedding 2 does not use the older `task_type` parameter seen in `gemini-embedding-001` examples.


In [4]:
test_text = "Machine learning is fascinating."
formatted_test_query = f"task: question answering | query: {test_text}"

embedding_response = gemini_client.models.embed_content(
    model=GEMINI_EMBEDDING_MODEL,
    contents=formatted_test_query,
)

test_vector = embedding_response.embeddings[0].values
if not test_vector:
    raise ValueError("Gemini returned an empty embedding.")

print("Input text:", test_text)
print("Vector dimensions:", len(test_vector))
print("First 5 values:", list(test_vector[:5]))

Input text: Machine learning is fascinating.
Vector dimensions: 3072
First 5 values: [-0.007061234, -0.002659985, 0.013159636, 0.027467655, -0.007062621]


## 6. Initialize Groq for generation only

Embeddings support retrieval; a chat model generates a natural-language answer. Keeping providers separate makes the system easier to teach and debug.

### Model ID note

This notebook uses the Groq-hosted model `qwen/qwen3.8-27b` through `ChatGroq`. The API provider is **Groq**; no OpenAI client or key is used. Groq currently labels this model as **Preview**, so treat it as a learning default rather than a production commitment. If it is not enabled for your account, change `GROQ_MODEL` in the next code cell to an available Groq model; re-check current model status before deployment.


In [5]:
GROQ_MODEL = "qwen/qwen3.8-27b"

groq_llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0.2,
)

print("Groq chat model initialized.")
print("Groq model ID:", GROQ_MODEL)

Groq chat model initialized.
Groq model ID: qwen/qwen3.8-27b


## 7. Test Groq generation independently

This test sends a prompt directly to Groq. It does **not** use Gemini embeddings, ChromaDB, or retrieval. If it fails, first check the Groq key, whether the configured model is enabled for your account, quota, and network.


In [6]:
test_message = groq_llm.invoke(
    "Explain RAG in simple language in two sentences."
)
print(test_message.content)

RAG (Retrieval-Augmented Generation) is a technique that allows AI to look up specific information from a database or documents before answering your question. This helps the AI provide more accurate and up-to-date answers by grounding its responses in real facts rather than just relying on its pre-trained knowledge.


## 8. Create the sample documents

A RAG system needs a knowledge source. We use the instructor's three compact documents so results are easy to inspect: Machine Learning, Deep Learning and Neural Networks, and Natural Language Processing (NLP). A controlled corpus is helpful while learning because we can tell whether the expected topic was retrieved.


In [7]:
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning finds patterns in unlabeled data. Reinforcement learning learns through
    interaction with an environment using rewards and penalties.
    """,

    """
    Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
    excel at sequential data processing.
    """,

    """
    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,
    machine translation, and question answering. Modern NLP heavily relies on transformer
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand
    context and relationships between words in text.
    """,
]

print("Number of sample documents:", len(sample_docs))
for i, text in enumerate(sample_docs, start=1):
    print(f"\nDocument {i}\n{'-' * 40}\n{text.strip()}")

Number of sample documents: 3

Document 1
----------------------------------------
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning finds patterns in unlabeled data. Reinforcement learning learns through
    interaction with an environment using rewards and penalties.

Document 2
----------------------------------------
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and

## 9. Save the documents as text files

The data already exists as Python strings, but writing files gives us the same loading workflow used with real corpora. This also fixes the instructor notebook's path mismatch: the old code wrote files to a temporary/current directory but tried to load them from a separate `data` directory.

We use one stable, project-local folder and load from that exact folder in the next section.


In [8]:
DOCS_DIR = Path("data/chromadb_gemini_groq_docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

for i, text in enumerate(sample_docs):
    file_path = DOCS_DIR / f"doc_{i}.txt"
    file_path.write_text(text.strip(), encoding="utf-8")
    print("Saved:", file_path)

print("Sample data directory:", DOCS_DIR.resolve())

Saved: data\chromadb_gemini_groq_docs\doc_0.txt
Saved: data\chromadb_gemini_groq_docs\doc_1.txt
Saved: data\chromadb_gemini_groq_docs\doc_2.txt
Sample data directory: E:\RAG\Vector_stores_&_Vector_databases\data\chromadb_gemini_groq_docs


## 10. Load the documents into LangChain

A LangChain `Document` holds two key fields: `page_content` (the text) and `metadata` (attributes such as source path, topic, and later chunk information).

`DirectoryLoader` selects files by path and glob; `TextLoader` reads each file as UTF-8. We attach a `topic` metadata value so the later filtering example demonstrates a real use case.


In [9]:
topic_by_stem = {
    "doc_0": "machine_learning",
    "doc_1": "deep_learning",
    "doc_2": "nlp",
}

loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documents = sorted(
    loader.load(),
    key=lambda doc: doc.metadata.get("source", ""),
)
if not documents:
    raise FileNotFoundError(f"No .txt documents found in {DOCS_DIR.resolve()}")

for doc in documents:
    source_stem = Path(doc.metadata.get("source", "unknown")).stem
    doc.metadata["topic"] = topic_by_stem.get(source_stem, "unknown")

print("Loaded documents:", len(documents))
print("First document preview:", documents[0].page_content[:200] + "...")
print("First document metadata:", documents[0].metadata)

Loaded documents: 3
First document preview: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are ...
First document metadata: {'source': 'data\\chromadb_gemini_groq_docs\\doc_0.txt', 'topic': 'machine_learning'}


## 11. Split documents into chunks

A **chunk** is a smaller segment of a source document used as a retrieval unit. If a whole long document is represented by one vector, a query about one detail may retrieve an overly broad passage. Chunks make retrieval more focused, though chunks that are too small can lose context.

- `chunk_size=500`: target chunk length in **characters**, not tokens.
- `chunk_overlap=50`: repeated text between neighbouring chunks to reduce boundary loss.
- Recursive separators try paragraph breaks, line breaks, spaces, and finally individual characters. This is more robust than `separators=[" "]`, which can split text awkwardly.


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
    add_start_index=True,
)

chunks = text_splitter.split_documents(documents)
if not chunks:
    raise ValueError("No chunks were created; check the loaded documents.")

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print("First chunk preview:", chunks[0].page_content[:200] + "...")
print("First chunk metadata:", chunks[0].metadata)

Created 6 chunks from 3 documents
First chunk preview: Machine Learning Fundamentals...
First chunk metadata: {'source': 'data\\chromadb_gemini_groq_docs\\doc_0.txt', 'topic': 'machine_learning', 'start_index': 0}


## 12. Wrap Gemini in LangChain's Embeddings interface

The direct API works, but LangChain's vector-store integration expects a standard embedding object with `embed_documents(texts)` and `embed_query(text)`. `GeminiEmbeddings` is the adapter that bridges these interfaces.

For Gemini Embedding 2 text-only question-answering retrieval, document text is formatted as `title: none | text: ...`, and query text is formatted as `task: question answering | query: ...`. The formats are not interchangeable. `embed_documents` processes each chunk separately to ensure each chunk receives its own vector rather than a single aggregated vector for a list of texts.


In [11]:
class GeminiEmbeddings(Embeddings):
    """Adapt Gemini Embedding 2 to LangChain's Embeddings interface."""

    def __init__(self, client, model):
        self.client = client
        self.model = model

    def _embed_one(self, text, *, is_query):
        if not isinstance(text, str) or not text.strip():
            raise ValueError("Embedding input must be a non-empty string.")

        if is_query:
            formatted_text = f"task: question answering | query: {text}"
        else:
            formatted_text = f"title: none | text: {text}"

        response = self.client.models.embed_content(
            model=self.model,
            contents=formatted_text,
        )
        values = response.embeddings[0].values
        if not values:
            raise ValueError("Gemini returned an empty embedding.")
        return [float(value) for value in values]

    def embed_documents(self, texts):
        return [self._embed_one(text, is_query=False) for text in texts]

    def embed_query(self, text):
        return self._embed_one(text, is_query=True)


gemini_embeddings = GeminiEmbeddings(
    client=gemini_client,
    model=GEMINI_EMBEDDING_MODEL,
)
print("Gemini embedding adapter created.")

Gemini embedding adapter created.


## 13. Test the LangChain embedding adapter

This is similar to the direct API test, but now we use `embed_query()` on our custom adapter. ChromaDB will later call these same interface methods automatically. This cell makes sure the adapter returns a list of floats with a non-zero number of dimensions.


In [12]:
adapter_test_vector = gemini_embeddings.embed_query(
    "What are the types of machine learning?"
)

print("Vector dimensions:", len(adapter_test_vector))
print("Vector type:", type(adapter_test_vector).__name__)
print("First 5 values:", adapter_test_vector[:5])

Vector dimensions: 3072
Vector type: list
First 5 values: [-0.010345737, -0.01155475, -0.0070516444, 0.02081251, -0.009954576]


## 14. Add a persistent embedding cache

### What is an embedding cache?

An **embedding cache** saves computed vectors so repeated input text can reuse the vector instead of making another provider request. This is different from ChromaDB: the cache avoids repeated embedding computation; ChromaDB stores the searchable corpus.

The original custom notebook had two near-duplicate single-vector cache cells, a separate chunk JSON cache, and repeated Chroma initialization. We consolidate that idea into LangChain's cache-backed embedding wrapper. The wrapper is used by ChromaDB itself, so cached embeddings are actually on the retrieval path.

If you change the embedding model or Gemini's input-prefix strategy, update the cache namespace (version suffix) so old vectors are not reused under new settings.


In [13]:
EMBEDDING_CACHE_DIR = Path("data/gemini_embedding_cache")
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

embedding_cache_store = LocalFileStore(str(EMBEDDING_CACHE_DIR))
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    gemini_embeddings,
    embedding_cache_store,
    namespace="gemini-embedding-2-question-answering-prefix-v1",
    query_embedding_cache=True,
)

print("Persistent embedding cache configured at:", EMBEDDING_CACHE_DIR.resolve())

Persistent embedding cache configured at: E:\RAG\Vector_stores_&_Vector_databases\data\gemini_embedding_cache


e:\RAG\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


## 15. Check the cache-backed embedding object

This cell requests the same query twice. The first call may ask Gemini for the vector; the second should be served from the cache when the first value has been stored. There is no guaranteed output difference because we print only vector dimensions, but API logs/latency or inspecting the cache directory can help verify reuse.


In [14]:
cache_test_query = "What is natural language processing?"
first_cached_vector = cached_embeddings.embed_query(cache_test_query)
second_cached_vector = cached_embeddings.embed_query(cache_test_query)

assert len(first_cached_vector) == len(second_cached_vector)
assert first_cached_vector == second_cached_vector
print("Repeated query embeddings match.")
print("Dimensions:", len(second_cached_vector))

Repeated query embeddings match.
Dimensions: 3072


## 16. Store chunks and embeddings in ChromaDB

A **vector store** keeps vectors with the associated text and metadata and offers similarity-search methods. ChromaDB provides persistent local storage for this teaching example.

We pass the cache-backed Gemini embedding object to `Chroma.from_documents()`. LangChain asks it to embed the chunks, then Chroma stores IDs, document text, metadata, and vectors. Explicit stable IDs prevent duplicate IDs when the same cells are rerun. We choose cosine distance explicitly so the later score explanation matches the configured metric.

`persist_directory` points to a dedicated folder. To make the “add a new document” lesson deterministic when students run all cells more than once, this teaching notebook deletes only its own named collection at initialization (`RESET_COLLECTION_ON_RUN = True`) and recreates it from the sample corpus. The data remains persisted on disk after the run. Set the flag to `False` if you deliberately want to reuse the existing collection; then inspect old records and IDs before comparing results.


In [15]:
PERSIST_DIRECTORY = Path("chroma_db_rag_gemini_groq_clean")
COLLECTION_NAME = "ml_knowledge_base_gemini_groq"
RESET_COLLECTION_ON_RUN = True
PERSIST_DIRECTORY.mkdir(parents=True, exist_ok=True)

# A PersistentClient stores the collection in this local directory.
chroma_client = chromadb.PersistentClient(path=str(PERSIST_DIRECTORY))
if RESET_COLLECTION_ON_RUN:
    try:
        chroma_client.delete_collection(name=COLLECTION_NAME)
        print("Deleted the previous tutorial collection for a clean run.")
    except chromadb.errors.NotFoundError:
        # Expected on the very first run, when the collection does not exist.
        pass

chunk_ids = [f"sample-chunk-{i:04d}" for i in range(len(chunks))]

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=cached_embeddings,
    ids=chunk_ids,
    client=chroma_client,
    collection_name=COLLECTION_NAME,
    collection_metadata={"hnsw:space": "cosine"},
)

print("ChromaDB collection initialized.")
print("Collection:", COLLECTION_NAME)
print("Persistence directory:", PERSIST_DIRECTORY.resolve())
print("Source chunks submitted:", len(chunks))

ChromaDB collection initialized.
Collection: ml_knowledge_base_gemini_groq
Persistence directory: E:\RAG\Vector_stores_&_Vector_databases\chroma_db_rag_gemini_groq_clean
Source chunks submitted: 6


## 17. Verify the stored records

A successful constructor message is not enough. Verify that the expected number of IDs, text records, and metadata records are present. Use the public `vectorstore.get()` wrapper instead of reaching into the integration's private `_collection` attribute.


In [16]:
stored_records = vectorstore.get(include=["documents", "metadatas"])

print("Stored vector records:", len(stored_records["ids"]))
print("First stored ID:", stored_records["ids"][0])
print("First document preview:", stored_records["documents"][0][:200] + "...")
print("First metadata:", stored_records["metadatas"][0])

Stored vector records: 6
First stored ID: sample-chunk-0000
First document preview: Machine Learning Fundamentals...
First metadata: {'start_index': 0, 'topic': 'machine_learning', 'source': 'data\\chromadb_gemini_groq_docs\\doc_0.txt'}


## 18. Search by semantic similarity

**Semantic search** retrieves text by comparing vector representations, which can find related ideas even when the query and document do not use exactly the same words.

The query is passed to the configured embedding interface, which uses Gemini to create a query vector. Chroma compares it with stored document vectors and returns the nearest chunks. `k=3` asks for at most three results; it does not guarantee that all three are relevant.


In [17]:
def show_similarity_results(query, k=3):
    results = vectorstore.similarity_search(query, k=k)
    print(f"\nQuery: {query}")
    print(f"Top results returned: {len(results)}")
    for i, doc in enumerate(results, start=1):
        print(f"\n--- Result {i} ---")
        print(doc.page_content[:200] + "...")
        print("Source:", doc.metadata.get("source", "Unknown"))
        print("Topic:", doc.metadata.get("topic", "Unknown"))
    return results


search_queries = [
    "What are the types of machine learning?",
    "What is NLP?",
    "What is Deep Learning?",
]

similarity_results = {
    query: show_similarity_results(query, k=3)
    for query in search_queries
}


Query: What are the types of machine learning?
Top results returned: 3

--- Result 1 ---
Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine lea...
Source: data\chromadb_gemini_groq_docs\doc_0.txt
Topic: machine_learning

--- Result 2 ---
Machine Learning Fundamentals...
Source: data\chromadb_gemini_groq_docs\doc_0.txt
Topic: machine_learning

--- Result 3 ---
Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning h...
Source: data\chromadb_gemini_groq_docs\doc_1.txt
Topic: deep_learning

Query: What is NLP?
Top results returned: 3

--- Result 1 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP inc

## 19. Inspect similarity-search distances

`similarity_search_with_score()` returns `(Document, score)` pairs. With this collection configured for cosine distance, **smaller distance means closer vectors**. A distance is not a probability, confidence percentage, or direct measure of factual correctness. The numerical meaning depends on the collection's distance metric, which is why the metric was set explicitly above.


In [18]:
score_query = "What is Deep Learning?"
results_with_scores = vectorstore.similarity_search_with_score(
    score_query, k=3)

for doc, distance in results_with_scores:
    print(f"\nCosine distance: {distance:.4f}")
    print("Topic:", doc.metadata.get("topic", "Unknown"))
    print("Source:", doc.metadata.get("source", "Unknown"))
    print("Content:", doc.page_content[:150] + "...")


Cosine distance: 0.2099
Topic: deep_learning
Source: data\chromadb_gemini_groq_docs\doc_1.txt
Content: Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of l...

Cosine distance: 0.2383
Topic: deep_learning
Source: data\chromadb_gemini_groq_docs\doc_1.txt
Content: Deep Learning and Neural Networks...

Cosine distance: 0.3127
Topic: machine_learning
Source: data\chromadb_gemini_groq_docs\doc_0.txt
Content: Machine Learning Fundamentals...


## 20. Filter retrieval using metadata

Vector similarity answers “which vectors are close?” Metadata filtering adds a structured condition such as topic, tenant, or document type. This is useful when a search must be restricted to an allowed subset of the corpus.

The documents loaded earlier were assigned `topic` metadata. The next query uses that field to search only the Deep Learning topic. The filter value must match the stored metadata exactly.


In [19]:
filtered_query = "Which neural network architectures are mentioned?"
filtered_docs = vectorstore.similarity_search(
    filtered_query,
    k=3,
    filter={"topic": "deep_learning"},
)

print("Filtered matches:", len(filtered_docs))
for doc in filtered_docs:
    print("\nTopic:", doc.metadata.get("topic"))
    print(doc.page_content[:200] + "...")

Filtered matches: 2

Topic: deep_learning
Deep Learning and Neural Networks...

Topic: deep_learning
Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning h...


## 21. Convert the vector store into a retriever

A **vector store** is the storage/search component. A **retriever** is a standard interface that takes a query and returns relevant documents, hiding details about how they are stored or searched.

`as_retriever(search_kwargs={"k": 3})` defines a retrieval policy that returns up to three chunks. The instructor notebook used `search_kwarg` (singular), which is a typo; LangChain expects `search_kwargs` (plural).


In [20]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

retriever_test_docs = retriever.invoke("What is machine learning?")
print("Retriever returned:", len(retriever_test_docs), "documents")
print(retriever_test_docs[0].page_content[:200] + "...")

Retriever returned: 3 documents
Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine lea...


## 22. Create the RAG prompt template

A **prompt template** defines how retrieved context and a user question are presented to the generation model. `{context}` is filled with retrieved passages; `{input}` is filled with the user question.

The system instruction asks the model to answer from the provided context and admit when the documents do not contain the answer. This reduces unsupported answers, but prompt wording alone cannot guarantee correctness.


In [21]:
system_prompt = """
You are an assistant for question-answering tasks.
Use the retrieved context to answer the user's question.
If the context does not contain the answer, say you do not know
based on the provided information. Do not invent facts.
Keep the answer concise, with a maximum of three sentences.

Retrieved context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

print("RAG prompt template created successfully.")

RAG prompt template created successfully.


## 23. Create the document chain (`create_stuff_documents_chain`)

This helper connects a chat model and prompt. The word **stuff** means the retrieved documents are formatted and inserted together into the prompt's `{context}` placeholder, then the model generates an answer.

This step does not retrieve anything itself. We create the document chain first, then connect it to the retriever in the next section. The helper is retained to preserve the instructor's API lesson.


In [22]:
document_chain = create_stuff_documents_chain(
    groq_llm,
    prompt,
)

print("Document chain created successfully.")

Document chain created successfully.


## 24. Connect retrieval and generation (`create_retrieval_chain`)

`create_retrieval_chain` connects the retriever to the document chain. For a question, it retrieves relevant `Document` objects, supplies them as `context`, then invokes the document chain to generate an answer.

The result dictionary includes the generated `answer` and the retrieved `context`. These source documents are useful for debugging and for building source-aware application interfaces.


In [23]:
rag_chain = create_retrieval_chain(
    retriever,
    document_chain,
)

print("Complete RAG chain created successfully.")

Complete RAG chain created successfully.


## 25. Ask the complete RAG system a question

This is the first end-to-end check. The chain should retrieve relevant chunks from ChromaDB, pass them to Groq with the question, and return an answer plus the source context. Inspect both answer and documents: a fluent answer can still be wrong if retrieval produced poor evidence.


In [24]:
question = "What are the three types of machine learning?"
response = rag_chain.invoke({"input": question})

print("Question:", question)
print("\nGenerated answer:")
print(response["answer"])
print("\nRetrieved context:")
for i, doc in enumerate(response["context"], start=1):
    print(f"\n--- Source {i} ---")
    print(doc.page_content[:250] + "...")
    print("Metadata:", doc.metadata)

Question: What are the three types of machine learning?

Generated answer:
The three main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.

Retrieved context:

--- Source 1 ---
Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning,...
Metadata: {'start_index': 35, 'topic': 'machine_learning', 'source': 'data\\chromadb_gemini_groq_docs\\doc_0.txt'}

--- Source 2 ---
Machine Learning Fundamentals...
Metadata: {'topic': 'machine_learning', 'start_index': 0, 'source': 'data\\chromadb_gemini_groq_docs\\doc_0.txt'}

--- Source 3 ---
Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized 

## 26. Create a reusable RAG query function

Instead of copy-pasting `.invoke()` and context-printing code for every question, a helper function packages those steps into one reusable call. It returns the full result dictionary so later code can inspect the answer and sources programmatically.


In [25]:
def query_rag_modern(question):
    """Ask the RAG chain a question and print its answer and source context."""
    result = rag_chain.invoke({"input": question})

    print("Question:", question)
    print("-" * 60)
    print("Answer:", result["answer"])
    print("\nRetrieved context:")
    for i, doc in enumerate(result["context"], start=1):
        print(f"\n--- Source {i} ---")
        print(doc.page_content[:200] + "...")
        print("Metadata:", doc.metadata)
    return result


test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?",
]

for test_question in test_questions:
    query_rag_modern(test_question)
    print("\n" + "=" * 80 + "\n")

Question: What are the three types of machine learning?
------------------------------------------------------------
Answer: The three main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.

Retrieved context:

--- Source 1 ---
Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine lea...
Metadata: {'topic': 'machine_learning', 'source': 'data\\chromadb_gemini_groq_docs\\doc_0.txt', 'start_index': 35}

--- Source 2 ---
Machine Learning Fundamentals...
Metadata: {'topic': 'machine_learning', 'start_index': 0, 'source': 'data\\chromadb_gemini_groq_docs\\doc_0.txt'}

--- Source 3 ---
Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning h...
Metadata: {'so

## 27. Alternative RAG pipeline using LCEL

**LCEL (LangChain Expression Language)** composes runnables with the `|` operator. This provides a transparent view of the flow: retrieve documents, format them as context, pass context and question to the prompt, call Groq, and parse the response into a string.

This is an alternative implementation, not a second provider or a second Chroma collection. It reuses the same `retriever` and `groq_llm`, but keeps a **separate `custom_prompt`** to preserve the instructor's LCEL teaching example. The LCEL chain returns an answer string; use the classic `rag_chain` when you want an output dictionary that already includes retrieved source documents.


In [26]:
def format_docs(docs):
    """Join retrieved document text into the prompt's context string."""
    return "\n\n".join(doc.page_content for doc in docs)


print("Document formatting helper ready.")

Document formatting helper ready.


The instructor used a separate prompt for the LCEL example, so we preserve that distinction. This template uses `{context}` for retrieved passages and `{question}` for the original user query. Its variable names must match the keys created by the LCEL `RunnableParallel` in the next stage.


In [27]:
custom_prompt = ChatPromptTemplate.from_template("""
Use the following context to answer the question.
If you do not know the answer based on the context, say you do not know.
Do not invent facts.

Context:
{context}

Question:
{question}
""")

print("LCEL-specific prompt created.")

LCEL-specific prompt created.


Now that `format_docs()` and `custom_prompt` exist, compose the flow explicitly. `RunnableParallel` creates the `context` and `question` fields expected by the custom prompt. `RunnablePassthrough()` forwards the original question unchanged. This cell reuses the already-created `retriever` and `groq_llm`; it does not create a second database or provider client.


In [28]:
from langchain_core.runnables import RunnableParallel

rag_chain_lcel = (
    RunnableParallel({
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    })
    | custom_prompt
    | groq_llm
    | StrOutputParser()
)

print("LCEL RAG chain created successfully.")

LCEL RAG chain created successfully.


Invoke the LCEL pipeline once with a single question. Because this alternative ends in `StrOutputParser`, its return value is a string rather than the dictionary returned by `rag_chain`.


In [29]:
lcel_question = "What is deep learning?"
lcel_answer = rag_chain_lcel.invoke(lcel_question)

print("Question:", lcel_question)
print("\nAnswer:")
print(lcel_answer)

Question: What is deep learning?

Answer:
Based on the context provided, deep learning is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes.


The instructor also defined a small helper that consistently sends a string question to the LCEL chain. We keep that reusable pattern, but use the current `.invoke()` API directly and return the answer so later code can use it.


In [30]:
def query_rag_lcel(question):
    """Run one question through the LCEL chain and return the answer string."""
    print("Question:", question)
    print("-" * 50)
    answer = rag_chain_lcel.invoke(question)
    print("Answer:", answer)
    return answer

Use the same small test set as the classic chain. This is a repeatability smoke test, not a retrieval-quality benchmark. The helper returns only the generated text; use `rag_chain` when you need the retrieved source `Document` objects for debugging.


In [31]:
for test_question in test_questions:
    query_rag_lcel(test_question)
    print("\n" + "=" * 80 + "\n")

Question: What are the three types of machine learning?
--------------------------------------------------
Answer: Based on the context provided, the three main types of machine learning are:

1.  Supervised learning
2.  Unsupervised learning
3.  Reinforcement learning


Question: What is deep learning and how does it relate to neural networks?
--------------------------------------------------
Answer: Based on the context provided, deep learning is a subset of machine learning that is based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes.


Question: What are CNNs best used for?
--------------------------------------------------
Answer: Based on the context provided, Convolutional Neural Networks (CNNs) are particularly effective for **image processing**.




## 28. Add a new document to the existing vector store

A vector store is useful only if it can be updated when the knowledge base changes. This section preserves the instructor's reinforcement-learning example.

We create a LangChain `Document` with explicit metadata, split it using the same splitter, and call the public `vectorstore.add_documents()` method. Explicit IDs make the example repeatable: re-running the same insertion updates the same record IDs rather than relying on random IDs. The embedding wrapper generates or reuses the appropriate Gemini vectors.


In [32]:
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
 decisions by interacting with an environment. The agent receives rewards or penalties
 based on its actions and learns to maximize cumulative reward over time. Key concepts
 in RL include: states, actions, rewards, policies, and value functions. Popular RL
 algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and
 Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),
 robotics, and autonomous systems.
""".strip()

new_doc = Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"},
)

print("New document length:", len(new_doc.page_content))
print("Metadata:", new_doc.metadata)

New document length: 586
Metadata: {'source': 'manual_addition', 'topic': 'reinforcement_learning'}


This cell performs two operations: the splitter preserves the document metadata on every chunk, and `add_documents()` stores the chunks and their Gemini embeddings in the existing collection. Reusing stable IDs makes reruns of this insertion update the same records. The collection reset above ensures this lesson starts without stale reinforcement-learning records when `RESET_COLLECTION_ON_RUN=True`.


In [33]:
new_chunks = text_splitter.split_documents([new_doc])
new_chunk_ids = [
    f"reinforcement-learning-{i:04d}" for i in range(len(new_chunks))]

added_ids = vectorstore.add_documents(
    documents=new_chunks,
    ids=new_chunk_ids,
)

print("New chunks created:", len(new_chunks))
print("IDs added or updated:", len(added_ids))

New chunks created: 3
IDs added or updated: 3


## 29. Retrieve information from the newly added document

This query should now be answerable from the reinforcement-learning passage. It verifies that adding data changed the searchable corpus, not just a Python variable. Inspect the returned context to check that the new document is actually being retrieved.


In [34]:
new_question = "What are the key concepts in reinforcement learning?"
updated_result = query_rag_modern(new_question)

Question: What are the key concepts in reinforcement learning?
------------------------------------------------------------
Answer: The key concepts in reinforcement learning include states, actions, rewards, policies, and value functions.

Retrieved context:

--- Source 1 ---
Reinforcement learning (RL) is a type of machine learning where an agent learns to make
 decisions by interacting with an environment. The agent receives rewards or penalties
 based on its actions and...
Metadata: {'source': 'manual_addition', 'topic': 'reinforcement_learning', 'start_index': 34}

--- Source 2 ---
Reinforcement Learning in Detail...
Metadata: {'source': 'manual_addition', 'start_index': 0, 'topic': 'reinforcement_learning'}

--- Source 3 ---
interaction with an environment using rewards and penalties....
Metadata: {'start_index': 481, 'topic': 'machine_learning', 'source': 'data\\chromadb_gemini_groq_docs\\doc_0.txt'}


## 30. Conversational RAG with chat history

### Why ordinary retrieval can fail on follow-up questions

A standard retriever receives only the latest text. After a user asks “Tell me about Python,” a follow-up like “What are its main libraries?” is ambiguous without prior context.

A **history-aware retriever** first rewrites the new question into a standalone query using the chat history, then retrieves documents for the rewritten question. The answer chain receives the retrieved context and chat history.

In this notebook, `chat_history` is a Python list of `HumanMessage` and `AIMessage` objects. This is in-memory example state, not durable multi-user conversation storage. A deployed app needs thread/session persistence and isolation.


In [35]:
contextualize_q_system_prompt = """Given a chat history and the latest user question,
which might refer to earlier context, formulate a standalone question that can be
understood without the history. Do not answer the question. If it is already standalone,
return it unchanged."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

print("Question-reformulation prompt created.")

Question-reformulation prompt created.


The history-aware retriever accepts the recent `chat_history` and latest `input`. When history is present, the Groq model rewrites the follow-up into a standalone search question, then the ordinary Chroma retriever finds relevant chunks. It is a retrieval enhancement; it does not write the final answer.


In [36]:
history_aware_retriever = create_history_aware_retriever(
    groq_llm,
    retriever,
    contextualize_q_prompt,
)

print("History-aware retriever created.")

History-aware retriever created.


### The answer prompt is a separate stage

The retriever's reformulation prompt only makes the latest question searchable. It does not generate the final answer. We still need an answer prompt with `{context}` and the user's `{input}`; `MessagesPlaceholder("chat_history")` also lets the answer model see the conversation history.


In [37]:
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the retrieved context to answer the user's question.
If the context does not contain the answer, say you do not know based on the provided information.
Do not invent facts. Keep the answer concise, with a maximum of three sentences.

Retrieved context:
{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(groq_llm, qa_prompt)
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain,
)

print("Conversational RAG chain created.")

Conversational RAG chain created.


## 31. Start a conversation with an empty history

The first question establishes what the conversation is about. We pass an empty `chat_history` explicitly so the chain's required input is unambiguous.


In [38]:
chat_history = []
first_question = "What is machine learning?"

result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": first_question,
})

print("Q:", first_question)
print("A:", result1["answer"])
print("Sources:")
for doc in result1["context"]:
    print("-", doc.metadata.get("source", "Unknown"),
          "|", doc.metadata.get("topic", "Unknown"))

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It includes three main types: supervised learning, unsupervised learning, and reinforcement learning.
Sources:
- data\chromadb_gemini_groq_docs\doc_0.txt | machine_learning
- data\chromadb_gemini_groq_docs\doc_0.txt | machine_learning
- data\chromadb_gemini_groq_docs\doc_1.txt | deep_learning


### Store the exchange for the next turn

The application must explicitly keep the current user message and model answer if the next call should use them as history. LangChain message objects preserve whether each message came from the human or AI. This list exists only in the current notebook kernel.


In [39]:
chat_history.extend([
    HumanMessage(content=first_question),
    AIMessage(content=result1["answer"]),
])

print("Messages in history:", len(chat_history))
for message in chat_history:
    print(type(message).__name__ + ":", message.content[:120])

Messages in history: 2
HumanMessage: What is machine learning?
AIMessage: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience withou


## 32. Ask a follow-up question

The follow-up “What are its main types?” does not name machine learning. The history-aware retriever can use the prior turn to reformulate it into a standalone question before retrieving relevant chunks. The answer model then receives retrieved context and chat history.


In [40]:
follow_up_question = "What are its main types?"

result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": follow_up_question,
})

print("Q:", follow_up_question)
print("A:", result2["answer"])
print("Retrieved source topics:", [doc.metadata.get(
    "topic", "Unknown") for doc in result2["context"]])

Q: What are its main types?
A: The three main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.
Retrieved source topics: ['machine_learning', 'reinforcement_learning', 'machine_learning']


## 33. Troubleshooting checklist and interview definitions

| Term | Interview-ready definition |
|---|---|
| RAG | Retrieves external evidence and supplies it to a generative model at answer time. |
| Embedding | A dense numeric representation produced by an embedding model. |
| Chunk | A passage cut from a source document and used as a retrieval unit. |
| Vector store | A component that stores vectors and associated records and exposes similarity search. |
| Vector database | A database system that manages vector data and retrieval, often with persistence, filtering, metadata, and operational features. |
| Similarity search | Ranks stored vectors by a defined distance/similarity measure relative to a query vector. |
| Distance score | A metric-specific measure of separation; not a probability or factuality score. In this notebook cosine distance is used and lower is closer. |
| Retriever | A standard interface that accepts a query and returns relevant documents. |
| Metadata filter | A structured condition that limits which records can be searched. |
| `create_stuff_documents_chain` | Combines retrieved documents into a context prompt and invokes the LLM. |
| `create_retrieval_chain` | Connects a retriever to a document-answer chain and returns answer plus context. |
| LCEL | LangChain Expression Language, which composes runnable components into a pipeline. |
| History-aware retriever | Uses conversation history to rewrite an ambiguous follow-up before retrieving documents. |
| Embedding cache | Reuses prior vectors for identical cached inputs to reduce repeated provider requests. It is not the vector database. |

### If something fails

1. **Environment error:** verify `.env` location and variable names; never print values.
2. **Embedding/API error:** confirm Gemini model ID, permissions, quota, and network; test the direct embedding cell first.
3. **Groq error:** confirm `GROQ_API_KEY`, model access, and quota; test the Groq cell separately.
4. **No documents:** check the current working directory and `data/chromadb_gemini_groq_docs`.
5. **Poor retrieval:** inspect returned chunks, metadata, chunk size/overlap, query formatting, and the embedding model. A fluent answer cannot fix irrelevant retrieved context.
6. **Dimension/model mismatch:** never mix vectors produced by different embedding models or incompatible formatting. Re-embed the corpus and change the cache namespace.
7. **Duplicate/stale records:** use stable IDs, a dedicated collection, and deliberate update/delete logic when a real corpus changes.
8. **Conversational confusion:** inspect the history list and rewritten question; notebook-level lists are not production session memory.


## 34. Source coverage audit: instructor notebook + custom notebook

The instructor file has 93 cells (including blank cells and outputs); the custom file has 91 cells. This cleaned notebook merges their useful lesson content in an ordered, rerunnable form. This is a **concept/code-path coverage audit**, not a byte-for-byte copy of every historical cell output.

| Instructor cell range | Content preserved in this notebook | Changes made |
|---|---|---|
| 0–3 | Intro, RAG architecture, environment/import concepts | Provider description changed from OpenAI to Gemini embeddings + Groq generation; imports consolidated. |
| 4–14 | All three sample documents, save files, load files, inspect documents, split chunks | Duplicate file-writing cell removed; temp/current directory mismatch fixed; recursive separators improved. |
| 15–18 | Test an embedding and inspect vector values | OpenAI embedding/key cells replaced with a direct Gemini Embedding 2 request plus the LangChain adapter. |
| 19–29 | Create ChromaDB, persist chunks, test the three search questions, inspect scores and explain metric direction | Uses the dedicated `langchain-chroma` import, stable IDs, public APIs, and explicitly configured cosine distance. |
| 30–34 | Initialize/test an LLM; explain provider selection | OpenAI chat calls and generic `init_chat_model` swaps replaced by one isolated Groq model cell and smoke test. No OpenAI API is called. |
| 35–49 | Retriever, prompt, `create_stuff_documents_chain`, `create_retrieval_chain`, invoke response, query helper and three test questions | Fixes `search_kwarg` typo; uses one prompt and one stable `groq_llm` variable; removes duplicated chain construction. |
| 50–61 | LCEL chain, format retrieved docs, test queries and helper behavior | Preserves LCEL as a clearly labelled alternative without reinitializing providers. |
| 62–72 | Add reinforcement-learning document, split and insert it, query new content | Uses `add_documents` and explicit IDs instead of accessing private `vectorstore._collection`. |
| 73–83 | Conversational memory, `MessagesPlaceholder`, human/AI messages, first question and follow-up | Restores this instructor section as a complete sequential section; explains in-memory versus persistent conversation state. |
| 84–91 | Groq provider example and setup | Moved to the earlier dedicated Groq section; repeated Groq initialization and API-key echo-style cells consolidated. |
| 92 | Empty trailing cell | Removed. |

### Custom notebook additions retained or improved

- Stronger “what / why / how / observe” Markdown before the implementation steps.
- Separate Gemini embedding and Groq generation sections; one variable `groq_llm` is used throughout.
- Persistent embedding caching integrated into the embedding interface; duplicate cross-notebook JSON cache snippets removed.
- Public Chroma interfaces instead of private `_collection` access.
- Metadata topic filtering, explicit cosine-distance semantics, stable IDs, and troubleshooting/interview definitions.
- The custom notebook's Gemini **chat-generation** cells were removed because the requested architecture uses Gemini for embeddings and Groq for generation. They duplicated the LLM role and caused confusing `llm`/`groq_llm` variable differences.

### Provider-specific lines intentionally replaced

The instructor's `OpenAIEmbeddings`, `ChatOpenAI`, `OPENAI_API_KEY`, and OpenAI `init_chat_model` calls are **not retained as executable code** because the requested version must use Gemini + Groq only. Their teaching purpose is covered through Gemini embedding calls/adapters and the Groq chat model. Unused imports, output-only inspection cells, duplicate initializations, and cells that depended on stale execution order are also removed.

## Final execution checklist

- Run all cells top to bottom in one environment.
- Confirm both provider smoke tests pass.
- Confirm the loaded document/chunk counts look reasonable.
- Inspect similarity results before judging generated answers.
- Confirm the reinforcement-learning document is retrieved after insertion.
- Confirm the follow-up question retrieves machine-learning context from chat history.
- Keep `.env`, ChromaDB folders, and cache folders out of source control.
